In [15]:
import pandas as pd
import numpy as np
import re

In [16]:
df = pd.read_json('../data/raw/meta_Software.json', lines=True)

In [17]:
df.drop_duplicates(subset=['asin'], inplace=True)

In [18]:
empty_values = [[], '', {}, '</div>']

def replace_empty_with_nan(val):
    if val in empty_values:
        return np.nan
    # Also catch empty lists/dicts represented as strings if any
    if str(val).strip() in ['[]', '{}', '']:
        return np.nan
    return val

df = df.map(replace_empty_with_nan)

In [19]:
(df.isnull().sum() / len(df)) * 100

category             8.979158
tech1               98.784602
description         13.156800
fit                100.000000
title                0.027728
also_buy            89.292481
tech2               99.981515
brand                2.440039
feature             28.490226
rank                 8.835898
also_view           73.127224
main_cat             0.231064
similar_item        99.787421
date                96.247516
price               74.610657
asin                 0.000000
imageURL            56.901890
imageURLHighRes     56.901890
details             16.451777
dtype: float64

In [20]:
columns_to_drop = [
    'tech1', 'fit', 'tech2', 'similar_item', 'date', 'datem', 
    'imageURL', 'imageURLHighRes', 'high_res_images'
]

df.drop(columns=[col for col in columns_to_drop if col in df.columns], inplace=True)

In [21]:
df

,category,description,title,also_buy,brand,feature,rank,also_view,main_cat,price,asin,details
0,NaN,NaN,HOLT PHYSICS LESSON PRESENTATION CD-ROM QUICK ...,NaN,HOLT. RINEHART AND WINSTON,NaN,"25,550 in Software (",NaN,Software,.a-box-inner{background-color:#fff}#alohaBuyBo...,0030672120,NaN
1,NaN,"[, <b>Latin rhythms that will get your kids si...","Sing, Watch, &amp; Learn Spanish (DVD + Guide)...",NaN,McGraw Hill,NaN,"15,792 in Software (",NaN,Software,NaN,0071480935,NaN
2,NaN,[<b>Connect is the only integrated learning sy...,Connect with LearnSmart Access Card for Microb...,NaN,McGraw-Hill Science/Engineering/Math,NaN,"16,900 in Software (",NaN,Software,NaN,007329506X,NaN
3,NaN,NaN,LearnSmart Standalone Access Card for Prescott...,NaN,McGraw-Hill Education,NaN,"12,986 in Software (",NaN,Software,NaN,0073513458,NaN
4,NaN,[<i>Anatomy &amp; Physiology Revealed Cat</i> ...,Anatomy &amp; Physiology Revealed Student Acce...,"[0323394612, 0323227937, 1118527488]",McGraw-Hill Education,NaN,"14,861 in Software (",NaN,Software,$4.83,0073525758,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
26785,"[Software, Children's]",[<b>Features: </b><br>1. Beautiful and fabulou...,925 Sterling Silver Angel Wings Heart Pendant ...,"[B07B4B12JX, B008UX1WJ2]",17maimeng,[Material: Guaranteed 100% Real Silver+Platinu...,"1,389,844 in Clothing, Shoes & Jewelry (","[B008UX1WJ2, B0094JVCKU, B077J3DR69, B07J4Z659...","<img src=""https://images-na.ssl-images-amazon....",$24.00,B01HEFZJC2,NaN
26786,"[Software, Children's, Material: Guaranteed 10...",[<b>Features: </b><br>1. Beautiful and fabulou...,925 Sterling Silver Love Heart Opal Pendant Ne...,NaN,17maimeng,[Material: Guaranteed 100% Real Silver+Platinu...,"1,469,307 in Clothing, Shoes & Jewelry (","[B074M486S8, B01D4H965K, B077L7GGF4, B019D8X0W...","<img src=""https://images-na.ssl-images-amazon....",$23.20,B01HEFZKEE,NaN
26787,"[Software, Digital Software, Antivirus & Secur...",[<div>Mac Internet Security X9 contains two of...,Intego Mac Internet Security X9 - 1 Mac - 1 ye...,NaN,Intego,[Award-winning antivirus software to protect a...,"2,733 in Software (","[B01MF5MTWP, B015724B8M, B07CYFFH4H, B07CY54KL...",Software,$39.99,B01HF3G4BS,"{'Downloading:': 'Currently, this item is avai..."
26788,NaN,[VersaCheck X9 for QuickBooks 2016 DNA Secure ...,VersaCheck X9 Small and Medium Business 2016 3...,NaN,Diversified Productivity Solutions Ltd,NaN,"15,575 in Software (",NaN,Software,$24.39,B01HF41TKI,"{'Shipping Weight:': '1.3 pounds', 'ASIN:': 'B..."


In [22]:
def clean_price(val):
    if pd.isna(val):
        return np.nan
    val = str(val)
    # If it looks like a massive HTML block, it's a corrupted price; return NaN
    if len(val) > 50 or '{' in val or '<' in val:
        return np.nan
    
    # Extract just the digits and the decimal point
    price_match = re.search(r'(\d+\.\d+)', val)
    if price_match:
        return float(price_match.group(1))
    return np.nan

df['price'] = df['price'].map(clean_price)

In [24]:
def clean_rank(val):
    # If it's a list, grab the first element (the primary rank)
    if isinstance(val, list):
        if len(val) == 0:
            return np.nan
        val = val[0]
        
    # Now safe to check for NaN
    try:
        if pd.isna(val):
            return np.nan
    except ValueError:
        pass
        
    val = str(val).replace(',', '')
    
    rank_match = re.search(r'(\d+)', val)
    if rank_match:
        return int(rank_match.group(1))
    return np.nan


df['rank'] = df['rank'].map(clean_rank)

In [25]:
df

,category,description,title,also_buy,brand,feature,rank,also_view,main_cat,price,asin,details
0,NaN,NaN,HOLT PHYSICS LESSON PRESENTATION CD-ROM QUICK ...,NaN,HOLT. RINEHART AND WINSTON,NaN,25550.0,NaN,Software,NaN,0030672120,NaN
1,NaN,"[, <b>Latin rhythms that will get your kids si...","Sing, Watch, &amp; Learn Spanish (DVD + Guide)...",NaN,McGraw Hill,NaN,15792.0,NaN,Software,NaN,0071480935,NaN
2,NaN,[<b>Connect is the only integrated learning sy...,Connect with LearnSmart Access Card for Microb...,NaN,McGraw-Hill Science/Engineering/Math,NaN,16900.0,NaN,Software,NaN,007329506X,NaN
3,NaN,NaN,LearnSmart Standalone Access Card for Prescott...,NaN,McGraw-Hill Education,NaN,12986.0,NaN,Software,NaN,0073513458,NaN
4,NaN,[<i>Anatomy &amp; Physiology Revealed Cat</i> ...,Anatomy &amp; Physiology Revealed Student Acce...,"[0323394612, 0323227937, 1118527488]",McGraw-Hill Education,NaN,14861.0,NaN,Software,4.83,0073525758,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
26785,"[Software, Children's]",[<b>Features: </b><br>1. Beautiful and fabulou...,925 Sterling Silver Angel Wings Heart Pendant ...,"[B07B4B12JX, B008UX1WJ2]",17maimeng,[Material: Guaranteed 100% Real Silver+Platinu...,1389844.0,"[B008UX1WJ2, B0094JVCKU, B077J3DR69, B07J4Z659...","<img src=""https://images-na.ssl-images-amazon....",24.00,B01HEFZJC2,NaN
26786,"[Software, Children's, Material: Guaranteed 10...",[<b>Features: </b><br>1. Beautiful and fabulou...,925 Sterling Silver Love Heart Opal Pendant Ne...,NaN,17maimeng,[Material: Guaranteed 100% Real Silver+Platinu...,1469307.0,"[B074M486S8, B01D4H965K, B077L7GGF4, B019D8X0W...","<img src=""https://images-na.ssl-images-amazon....",23.20,B01HEFZKEE,NaN
26787,"[Software, Digital Software, Antivirus & Secur...",[<div>Mac Internet Security X9 contains two of...,Intego Mac Internet Security X9 - 1 Mac - 1 ye...,NaN,Intego,[Award-winning antivirus software to protect a...,2733.0,"[B01MF5MTWP, B015724B8M, B07CYFFH4H, B07CY54KL...",Software,39.99,B01HF3G4BS,"{'Downloading:': 'Currently, this item is avai..."
26788,NaN,[VersaCheck X9 for QuickBooks 2016 DNA Secure ...,VersaCheck X9 Small and Medium Business 2016 3...,NaN,Diversified Productivity Solutions Ltd,NaN,15575.0,NaN,Software,24.39,B01HF41TKI,"{'Shipping Weight:': '1.3 pounds', 'ASIN:': 'B..."


In [26]:
(df.isnull().sum() / len(df)) * 100

category        8.979158
description    13.156800
title           0.027728
also_buy       89.292481
brand           2.440039
feature        28.490226
rank            8.835898
also_view      73.127224
main_cat        0.231064
price          78.728222
asin            0.000000
details        16.451777
dtype: float64

In [27]:
df.to_csv('../data/clean_reviews.csv', index=False)

In [28]:
df.to_parquet('../data/clean_meta.parquet', index=False)